# 🧠 Ejercicio 1: NER con SpaCy

En este ejercicio implementaremos un modelo de Reconocimiento de Entidades Nombradas (NER) usando la librería spaCy.

Objetivos:
- Cargar un modelo preentrenado
- Procesar texto
- Identificar entidades como personas, organizaciones y lugares

In [21]:
import sys
!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 4.4 MB/s eta 0:00:00m eta 0:00:010:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
import spacy

# Cargar el modelo de lenguaje de SpaCy
nlp = spacy.load("en_core_web_sm")

# Texto de ejemplo
text = "Elon Musk founded SpaceX in 2002 in California."

# Aplicar el modelo de NER
doc = nlp(text)

# Imprimir las entidades reconocidas
for ent in doc.ents:
    print(ent.text, "-", ent.label_)



Elon Musk - PERSON
2002 - DATE
California - GPE


### 📌 Resultado esperado

- Elon Musk → PERSON
- SpaceX → ORG
- 2002 → DATE
- California → GPE

Estas etiquetas corresponden a:
- PERSON: Persona
- ORG: Organización
- DATE: Fecha
- GPE: Lugar geopolítico

# 🤖 Ejercicio 2: NER con Hugging Face (BERT)

En este ejercicio implementaremos un modelo de Reconocimiento de Entidades Nombradas (NER) usando transformers.

Objetivos:
- Usar un modelo preentrenado basado en BERT
- Aplicar NER sobre texto
- Comparar resultados con SpaCy

In [26]:
from transformers import pipeline

In [ ]:
# Cargar el pipeline de NER
ner_pipeline = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"  # Agrupa tokens en entidades completas
)

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [28]:
# Texto de ejemplo
text = "Elon Musk founded Tesla in 2003 in Palo Alto."

# Aplicar el modelo de NER
ner_results = ner_pipeline(text)

# Imprimir los resultados
for entity in ner_results:
    print(f"Entidad: {entity['word']}, Tipo: {entity['entity_group']}, Score: {entity['score']:.4f}")

Entidad: Elon Musk, Tipo: PER, Score: 0.9978
Entidad: Tesla, Tipo: ORG, Score: 0.9487
Entidad: Palo Alto, Tipo: LOC, Score: 0.9960


### 📌 Resultado esperado

- Elon Musk → PER (Persona)
- Tesla → ORG (Organización)
- 2003 → DATE
- Palo Alto → LOC (Lugar)

El score indica la confianza del modelo.

# 📊 Ejercicio 3: Evaluación de modelos NER

En este ejercicio aprenderemos a evaluar modelos de Reconocimiento de Entidades Nombradas (NER).

Objetivos:
- Calcular precisión, recall y F1-score
- Analizar el rendimiento de un modelo
- Comparar etiquetas reales vs predichas

In [30]:
from sklearn.metrics import classification_report

def evaluate_ner(true_labels, pred_labels):
    print(classification_report(true_labels, pred_labels))

# Etiquetas reales
true_labels = ["O", "B-PER", "I-PER", "O", "B-ORG", "O"]

# Etiquetas predichas
pred_labels = ["O", "B-PER", "I-PER", "O", "O", "O"]

#evaluar el modelo
evaluate_ner(true_labels, pred_labels)

              precision    recall  f1-score   support

       B-ORG       0.00      0.00      0.00         1
       B-PER       1.00      1.00      1.00         1
       I-PER       1.00      1.00      1.00         1
           O       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.69      0.75      0.71         6
weighted avg       0.71      0.83      0.76         6



/home/jdvalmart/MachineDeepLearning/venv_tf/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jdvalmart/MachineDeepLearning/venv_tf/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jdvalmart/MachineDeepLearning/venv_tf/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

### 📌 Métricas

- Precision: Qué tan correctas son las predicciones positivas
- Recall: Qué tantas entidades reales fueron detectadas
- F1-score: Balance entre precisión y recall

El F1-score es la métrica más importante en NER.

# ⚙️ Ejercicio 4: Pipeline de NER con procesamiento paralelo

En este ejercicio integraremos un modelo de NER en un pipeline de procesamiento de datos.

Objetivos:
- Procesar grandes volúmenes de texto
- Dividir datos en fragmentos (chunks)
- Aplicar NER en paralelo

In [31]:
import spacy
from concurrent.futures import ProcessPoolExecutor

In [32]:
nlp = spacy.load("en_core_web_sm")

In [33]:
def process_text_chunk(text_chunk):
    doc = nlp(text_chunk)
    return [(ent.text, ent.label_) for ent in doc.ents]

In [34]:
text_chunks = [
    "Elon Musk founded SpaceX.",
    "Tesla is based in Palo Alto.",
    "Google was founded in California.",
    "Microsoft acquired GitHub in 2018."
]

In [35]:
with ProcessPoolExecutor() as executor:
    results = executor.map(process_text_chunk, text_chunks)

for result in results:
    print(result)

[('Elon Musk', 'PERSON')]
[('Tesla', 'ORG'), ('Palo Alto', 'GPE')]
[('Google', 'ORG'), ('California', 'GPE')]
[('Microsoft', 'ORG'), ('GitHub', 'ORG'), ('2018', 'DATE')]


### 📌 ¿Qué está pasando?

- Dividimos el texto en fragmentos (chunks)
- Cada fragmento se procesa de forma independiente
- Se ejecuta en paralelo usando múltiples procesos

Esto mejora el rendimiento cuando se trabaja con grandes volúmenes de datos.